# NBA DFS Season Backtest - Per Player Models with Benchmark Comparison

Walk-forward backtesting across multiple slates using per-player XGBoost models and season average benchmark comparison.

## Methodology

1. Load historical data for training
2. Load injury data per slate for injury status features
3. Build features using YAML-configured rolling statistics and injury features
4. Train separate XGBoost model per player on historical data (with optional GPU acceleration)
5. Calculate season average benchmark for comparison
6. Walk forward through test period, generating predictions for each slate
7. Compare predictions to actual results
8. Analyze model vs benchmark performance overall and by salary tier
9. Statistical significance testing across all slates

## GPU Acceleration

This notebook supports GPU-accelerated training with XGBoost 2.0+:
- Set `USE_GPU = True` in the configuration cell to enable GPU training
- Optionally specify `MODEL_CONFIG_PATH` to load optimized hyperparameters from YAML
- XGBoost will automatically use CUDA for faster model training
- Recommended for large-scale backtests with many per-player models

GPU configuration example:
```python
USE_GPU = True
GPU_ID = 0  # First GPU
MODEL_CONFIG_PATH = 'config/models/xgboost_a100.yaml'  # Optional: GPU-tuned params
```

## Injury Data Integration

Injury data is automatically loaded per slate and merged with player features:
- **injury_status**: Healthy, Out, Questionable, Doubtful, Day-To-Day
- **is_injured**: Binary flag (1 if any injury designation)
- **is_out**: Binary flag (1 if ruled out)
- **is_questionable**: Binary flag (1 if questionable)
- **is_doubtful**: Binary flag (1 if doubtful)
- **is_day_to_day**: Binary flag (1 if day-to-day)
- **injury_designation**: Original injury designation text
- **injury_description**: Injury details from API

Injury features are configured in `config/features/default_features.yaml` and `config/features/base_features.yaml` via the `InjuryTransformer`.

## Player Filtering

Filter players before model training and projection. Multiple filter types can be combined (AND logic):

### Salary Filtering
- **FILTER_SALARY_MIN**: Set to int to include only players above this salary (e.g., 5000 for $5k+)
- **FILTER_SALARY_MAX**: Set to int to include only players below this salary (e.g., 10000 for up to $10k)

### Injury Filtering
- **FILTER_EXCLUDE_OUT**: Set to True to exclude OUT players
- **FILTER_EXCLUDE_DOUBTFUL**: Set to True to exclude DOUBTFUL players
- **FILTER_EXCLUDE_QUESTIONABLE**: Set to True to exclude QUESTIONABLE players

### Player ID/Name Filtering
- **FILTER_PLAYER_IDS**: Filter by player IDs. Supports:
  - List format: `[201935, 2544, 201950]`
  - Comma-separated string: `"201935,2544,201950"`
  - Space-separated string: `"201935 2544 201950"`
  - Single ID: `201935`
  
- **FILTER_PLAYER_NAMES**: Filter by player names (partial match, case-insensitive). Supports:
  - List format: `['LeBron', 'Durant']`
  - Comma-separated string: `"LeBron,Durant"`
  - Space-separated string: `"LeBron Durant"`
  - Single name: `"LeBron"`

- **FILTER_PLAYERS_CSV**: Filter by player IDs from a CSV file. CSV must contain a 'playerID' column.
  - Example: `"my_players.csv"` or `"/path/to/players.csv"`

Filters apply to both training and projection phases.

## Setup

In [1]:
!pip install plotly

In [2]:
!pip install cupy

In [3]:
import sys
from pathlib import Path
import logging
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from scipy import stats

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.walk_forward_backtest import WalkForwardBacktest
from src.data.loaders.historical_loader import HistoricalDataLoader

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print('Setup complete')

Setup complete


## Configuration

In [4]:
import psutil
import multiprocessing

cpu_count = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"CPU Cores: {cpu_count}")
print(f"RAM: {ram_gb:.1f} GB")
print(f"Recommended n_jobs: {cpu_count}")

if ram_gb < 12:
    print("WARNING: Low RAM detected. Consider reducing n_jobs or processing fewer players.")

CPU Cores: 32
RAM: 31.8 GB
Recommended n_jobs: 32


In [ ]:
DB_PATH = str(repo_root / 'nba_dfs.db')
OUTPUT_DIR = str(repo_root / 'data' / 'outputs')

TEST_START = '20250205'
TEST_END = '20250430'

NUM_SEASONS = 1
FEATURE_CONFIG = 'default_features'
MODEL_TYPE = 'xgboost'
MIN_PLAYER_GAMES = 10
MIN_GAMES_FOR_BENCHMARK = 5
RECALIBRATE_DAYS = 7
SALARY_TIERS = [0, 4000, 6000, 8000, 15000]

PER_PLAYER_MODELS = False
SAVE_MODELS = True
SAVE_PREDICTIONS = True
N_JOBS = 32

# GPU configuration
USE_GPU = True  # Set to True to enable GPU acceleration
GPU_ID = 0  # GPU device ID (0 for first GPU)
MODEL_CONFIG_PATH = "C:\\Users\\antho\\OneDrive\\Documents\\Repositories\\delapan-fantasy\\config\\models\\xgboost_default.yaml"  
# Path to GPU-optimized model config (e.g., 'config/models/xgboost_a100.yaml')

# Player filtering configuration - Salary and Injury filters
FILTER_SALARY_MIN = 5000  # Set to int to filter players below this salary
FILTER_SALARY_MAX = None  # Set to int to filter players above this salary
FILTER_EXCLUDE_OUT = False  # Set to True to exclude OUT players
FILTER_EXCLUDE_DOUBTFUL = False  # Set to True to exclude DOUBTFUL players
FILTER_EXCLUDE_QUESTIONABLE = False  # Set to True to exclude QUESTIONABLE players

# Player filtering configuration - ID and Name filters
FILTER_PLAYER_IDS = None  # Set to list of player IDs, e.g., [201935, 2544] or comma/space-separated string
FILTER_PLAYER_NAMES = ['Lebron James', 'Stephen Curry']  # Set to list of player names, e.g., ['LeBron', 'Durant'] or comma/space-separated string
FILTER_PLAYERS_CSV = None  # Set to CSV file path with playerID column, e.g., 'my_players.csv'

TRAIN_START = HistoricalDataLoader.get_season_start_date(TEST_START) if NUM_SEASONS == 1 else HistoricalDataLoader.get_previous_season_start_date(TEST_START)
TEST_END_DT = datetime.strptime(TEST_END, '%Y%m%d')
TRAIN_END = (TEST_END_DT - timedelta(days=1)).strftime('%Y%m%d')

# Load GPU-optimized model config if specified, otherwise use default params
if USE_GPU and MODEL_CONFIG_PATH:
    import yaml
    with open(repo_root / MODEL_CONFIG_PATH, 'r') as f:
        gpu_config = yaml.safe_load(f)
        MODEL_PARAMS = gpu_config.get('model', {}).get('params', {})
        # Ensure GPU device is set
        if 'device' not in MODEL_PARAMS:
            MODEL_PARAMS['device'] = f'cuda:{GPU_ID}'
        if 'tree_method' not in MODEL_PARAMS:
            MODEL_PARAMS['tree_method'] = 'hist'
else:
    MODEL_PARAMS = {
        'max_depth': 6,
        'learning_rate': 0.05,
        'n_estimators': 200,
        'min_child_weight': 5,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'objective': 'reg:squarederror',
        'random_state': 42
    }
    # Add GPU params if GPU enabled but no config file
    if USE_GPU:
        MODEL_PARAMS['device'] = f'cuda:{GPU_ID}'
        MODEL_PARAMS['tree_method'] = 'hist'

print('Configuration:')
print(f'  Database: {DB_PATH}')
print(f'  Output Directory: {OUTPUT_DIR}')
print(f'  Training Period: {TRAIN_START} to {TRAIN_END}')
print(f'  Testing Period: {TEST_START} to {TEST_END}')
print(f'  Number of Seasons: {NUM_SEASONS}')
print(f'  Model Type: {MODEL_TYPE}')
print(f'  Feature Config: {FEATURE_CONFIG}')
print(f'  Per-Player Models: {PER_PLAYER_MODELS}')
print(f'  Min Player Games: {MIN_PLAYER_GAMES}')
print(f'  Min Benchmark Games: {MIN_GAMES_FOR_BENCHMARK}')
print(f'  Recalibrate Every: {RECALIBRATE_DAYS} days')
print(f'  Parallel Jobs: {N_JOBS} ({"all cores" if N_JOBS == -1 else "sequential" if N_JOBS == 1 else f"{N_JOBS} workers"})')
print(f'  Save Models: {SAVE_MODELS}')
print(f'  Save Predictions: {SAVE_PREDICTIONS}')
print(f'  Salary Tiers: {SALARY_TIERS}')

if USE_GPU:
    print(f'\n  GPU Configuration:')
    print(f'    Enabled: Yes')
    print(f'    GPU ID: {GPU_ID}')
    print(f'    Device: {MODEL_PARAMS.get("device", "N/A")}')
    print(f'    Tree Method: {MODEL_PARAMS.get("tree_method", "N/A")}')
    if MODEL_CONFIG_PATH:
        print(f'    Config File: {MODEL_CONFIG_PATH}')
else:
    print(f'\n  GPU Configuration: Disabled (CPU mode)')

print(f'\n  Player Filters:')
print(f'    Salary & Injury:')
if FILTER_SALARY_MIN:
    print(f'      - Minimum salary: ${FILTER_SALARY_MIN}')
if FILTER_SALARY_MAX:
    print(f'      - Maximum salary: ${FILTER_SALARY_MAX}')
if FILTER_EXCLUDE_OUT:
    print(f'      - Exclude OUT players')
if FILTER_EXCLUDE_DOUBTFUL:
    print(f'      - Exclude DOUBTFUL players')
if FILTER_EXCLUDE_QUESTIONABLE:
    print(f'      - Exclude QUESTIONABLE players')

print(f'\n    Player ID/Name:')
if FILTER_PLAYER_IDS:
    if isinstance(FILTER_PLAYER_IDS, str):
        ids_list = [id.strip() for id in FILTER_PLAYER_IDS.replace(',', ' ').split()]
    else:
        ids_list = FILTER_PLAYER_IDS if isinstance(FILTER_PLAYER_IDS, list) else [FILTER_PLAYER_IDS]
    ids_display = ', '.join(str(id) for id in ids_list[:3])
    if len(ids_list) > 3:
        ids_display += f", ... (+{len(ids_list)-3} more)"
    print(f'      - Filter by player IDs: {ids_display}')
if FILTER_PLAYER_NAMES:
    if isinstance(FILTER_PLAYER_NAMES, str):
        names_list = [name.strip() for name in FILTER_PLAYER_NAMES.replace(',', '|').split('|')]
    else:
        names_list = FILTER_PLAYER_NAMES if isinstance(FILTER_PLAYER_NAMES, list) else [FILTER_PLAYER_NAMES]
    names_display = ', '.join(names_list[:2])
    if len(names_list) > 2:
        names_display += f", ... (+{len(names_list)-2} more)"
    print(f'      - Filter by player names: {names_display}')
if FILTER_PLAYERS_CSV:
    print(f'      - Filter from CSV file: {FILTER_PLAYERS_CSV}')

if not any([FILTER_SALARY_MIN, FILTER_SALARY_MAX, FILTER_EXCLUDE_OUT, FILTER_EXCLUDE_DOUBTFUL, 
            FILTER_EXCLUDE_QUESTIONABLE, FILTER_PLAYER_IDS, FILTER_PLAYER_NAMES, FILTER_PLAYERS_CSV]):
    print(f'      - No player filters configured')

Configuration:
  Database: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\nba_dfs.db
  Output Directory: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs
  Training Period: 20241001 to 20250429
  Testing Period: 20250205 to 20250430
  Number of Seasons: 1
  Model Type: xgboost
  Feature Config: default_features
  Per-Player Models: False
  Min Player Games: 10
  Min Benchmark Games: 5
  Recalibrate Every: 7 days
  Parallel Jobs: 32 (32 workers)
  Save Models: True
  Save Predictions: True
  Salary Tiers: [0, 4000, 6000, 8000, 15000]

  GPU Configuration:
    Enabled: Yes
    GPU ID: 0
    Device: cuda:0
    Tree Method: hist
    Config File: C:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\config\models\xgboost_default.yaml

  Player Filters:
    Salary & Injury:
      - Minimum salary: $5000
      - Exclude OUT players
      - Exclude DOUBTFUL players

    Player ID/Name:
      - Filter by player names: Lebron James, Stephen Cur

## Run Walk-Forward Backtest

Initialize and run the WalkForwardBacktest class. This will:
1. Load historical training data
2. Build features using YAML-configured pipeline
3. Initialize season average benchmark
4. Train per-player models (or slate-level model)
5. Generate predictions for each test slate
6. Evaluate against actuals
7. Perform statistical analysis

In [6]:
from src.filters import ColumnFilter, InjuryFilter
from src.filters.player_filters import PlayerIDFilter, PlayerNameFilter, PlayerIDFromCSVFilter

# Build player filters from configuration
player_filters = []

# Salary filters
if FILTER_SALARY_MIN is not None:
    player_filters.append(ColumnFilter('salary', '>=', FILTER_SALARY_MIN))
    print(f'Added filter: salary >= {FILTER_SALARY_MIN}')

if FILTER_SALARY_MAX is not None:
    player_filters.append(ColumnFilter('salary', '<=', FILTER_SALARY_MAX))
    print(f'Added filter: salary <= {FILTER_SALARY_MAX}')

# Injury filters
if FILTER_EXCLUDE_OUT or FILTER_EXCLUDE_DOUBTFUL or FILTER_EXCLUDE_QUESTIONABLE:
    injury_filter = InjuryFilter(
        exclude_out=FILTER_EXCLUDE_OUT,
        exclude_doubtful=FILTER_EXCLUDE_DOUBTFUL,
        exclude_questionable=FILTER_EXCLUDE_QUESTIONABLE
    )
    player_filters.append(injury_filter)
    excluded = []
    if FILTER_EXCLUDE_OUT:
        excluded.append('OUT')
    if FILTER_EXCLUDE_DOUBTFUL:
        excluded.append('DOUBTFUL')
    if FILTER_EXCLUDE_QUESTIONABLE:
        excluded.append('QUESTIONABLE')
    print(f'Added filter: exclude injury status {", ".join(excluded)}')

# Player ID filters
if FILTER_PLAYER_IDS:
    # Parse comma or space-separated player IDs
    if isinstance(FILTER_PLAYER_IDS, str):
        player_ids = [pid.strip() for pid in FILTER_PLAYER_IDS.replace(',', ' ').split() if pid.strip()]
    else:
        player_ids = FILTER_PLAYER_IDS if isinstance(FILTER_PLAYER_IDS, list) else [FILTER_PLAYER_IDS]
    
    if player_ids:
        player_id_filter = PlayerIDFilter(player_ids)
        player_filters.append(player_id_filter)
        ids_display = ', '.join(str(pid) for pid in player_ids[:5])
        if len(player_ids) > 5:
            ids_display += f", ... (+{len(player_ids) - 5} more)"
        print(f'Added filter: player ID in [{ids_display}]')

# Player name filters
if FILTER_PLAYER_NAMES:
    # Parse comma or space-separated player names
    if isinstance(FILTER_PLAYER_NAMES, str):
        player_names = [name.strip() for name in FILTER_PLAYER_NAMES.replace(',', '|').split('|') if name.strip()]
    else:
        player_names = FILTER_PLAYER_NAMES if isinstance(FILTER_PLAYER_NAMES, list) else [FILTER_PLAYER_NAMES]
    
    if player_names:
        player_name_filter = PlayerNameFilter(player_names, case_sensitive=False)
        player_filters.append(player_name_filter)
        names_display = ', '.join(player_names[:3])
        if len(player_names) > 3:
            names_display += f", ... (+{len(player_names) - 3} more)"
        print(f'Added filter: player name contains [{names_display}]')

# Player ID from CSV file
if FILTER_PLAYERS_CSV:
    try:
        csv_filter = PlayerIDFromCSVFilter(FILTER_PLAYERS_CSV)
        player_filters.append(csv_filter)
        print(f'Added filter: player IDs from CSV ({len(csv_filter.player_ids)} players)')
    except FileNotFoundError as e:
        print(f'ERROR: {e}')
    except ValueError as e:
        print(f'ERROR: {e}')

if player_filters:
    print(f'\nTotal filters: {len(player_filters)}')
else:
    print('No player filters configured')

backtest = WalkForwardBacktest(
    db_path=DB_PATH,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    model_type=MODEL_TYPE,
    model_params=MODEL_PARAMS,
    feature_config=FEATURE_CONFIG,
    output_dir=OUTPUT_DIR,
    per_player_models=PER_PLAYER_MODELS,
    min_player_games=MIN_PLAYER_GAMES,
    min_games_for_benchmark=MIN_GAMES_FOR_BENCHMARK,
    recalibrate_days=RECALIBRATE_DAYS,
    num_seasons=NUM_SEASONS,
    salary_tiers=SALARY_TIERS,
    save_models=SAVE_MODELS,
    save_predictions=SAVE_PREDICTIONS,
    n_jobs=N_JOBS,
    player_filters=player_filters if player_filters else None
)

print('\nRunning backtest...')
results = backtest.run()

if 'error' in results:
    print(f"ERROR: {results['error']}")
else:
    print(f"\nBacktest completed successfully!")
    print(f"Processed {results['num_slates']} slates")
    print(f"Model MAPE: {results['model_mean_mape']:.2f}%")
    print(f"Benchmark MAPE: {results['benchmark_mean_mape']:.2f}%")
    print(f"Improvement: {results['mape_improvement']:+.2f}%")

2025-10-19 00:05:22,118 - src.walk_forward_backtest - WARNING - train_end (20250429) is after test_start (20250205). Training data will overlap with test window.
2025-10-19 00:05:22,119 - src.data.storage.sqlite_storage - INFO - Initialized SQLiteStorage with database: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\nba_dfs.db
2025-10-19 00:05:22,123 - src.utils.feature_config - INFO - Loaded feature config: Default Feature Set
2025-10-19 00:05:22,123 - src.utils.feature_config - INFO - Added RollingStatsTransformer: windows=[3, 5, 10], stats=21, include_std=True
2025-10-19 00:05:22,123 - src.utils.feature_config - INFO - Added EWMATransformer: span=5, stats=21
2025-10-19 00:05:22,124 - src.utils.feature_config - INFO - Added TargetTransformer: target_col=fpts, shift_periods=-1
2025-10-19 00:05:22,124 - src.utils.feature_config - INFO - Added InjuryTransformer
2025-10-19 00:05:22,125 - src.walk_forward_backtest - INFO - Initialized WalkForwardBacktest
2025-10-19 00:05:22

Added filter: salary >= 5000
Added filter: exclude injury status OUT, DOUBTFUL
Added filter: player name contains [Lebron James, Stephen Curry]

Total filters: 3

Running backtest...

Backtesting 76 slates from 20250205 to 20250430



2025-10-19 00:05:22,343 - src.data.loaders.historical_loader - INFO - Loaded 27183 player logs from 2024-10-22 00:00:00 to 2025-04-28 00:00:00
2025-10-19 00:05:22,344 - src.walk_forward_backtest - INFO - Loaded 27183 training records
2025-10-19 00:05:22,344 - src.walk_forward_backtest - INFO - Building features for benchmark...
2025-10-19 00:05:22,650 - src.walk_forward_backtest - INFO - Calculated fantasy points for training data
2025-10-19 00:05:35,488 - src.walk_forward_backtest - INFO - Generated 27183 feature rows with 147 features
2025-10-19 00:05:35,528 - src.walk_forward_backtest - INFO - Qualified players: 545 (min_games=5)
2025-10-19 00:05:35,528 - src.walk_forward_backtest - INFO - Initializing SeasonAverageBenchmark...
2025-10-19 00:05:35,532 - src.evaluation.benchmarks.season_average - INFO - Fitted benchmark for 545 players (min_games=5)
2025-10-19 00:05:35,532 - src.walk_forward_backtest - INFO - Benchmark fitted successfully for 545 players
2025-10-19 00:05:35,532 - src

ValueError: Required injury columns not found: ['is_out', 'is_doubtful']. Ensure InjuryTransformer has been applied to data.

## Extract Results

Extract the daily results and predictions dataframes from the backtest results.

In [ ]:
if 'error' not in results:
    results_df = results['daily_results']
    all_predictions_df = results['all_predictions']
    
    print('='*80)
    print('BACKTEST RESULTS SUMMARY')
    print('='*80)
    print(f'\nNumber of Slates: {results["num_slates"]}')
    print(f'Date Range: {results["date_range"]}')
    print(f'\nTotal Players Evaluated: {results["total_players_evaluated"]:.0f}')
    print(f'Average Players per Slate: {results["avg_players_per_slate"]:.1f}')
    print(f'\nModel Performance:')
    print(f'  Mean MAPE: {results["model_mean_mape"]:.2f}%')
    print(f'  Median MAPE: {results["model_median_mape"]:.2f}%')
    print(f'  Std MAPE: {results["model_std_mape"]:.2f}%')
    print(f'  Mean RMSE: {results["model_mean_rmse"]:.2f}')
    print(f'  Mean MAE: {results["model_mean_mae"]:.2f}')
    print(f'  Mean Correlation: {results["model_mean_correlation"]:.3f}')
    print(f'\nBenchmark Performance:')
    print(f'  Mean MAPE: {results["benchmark_mean_mape"]:.2f}%')
    print(f'  Median MAPE: {results["benchmark_median_mape"]:.2f}%')
    print(f'\nImprovement (Model vs Benchmark):')
    print(f'  MAPE Improvement: {results["mape_improvement"]:+.2f}%')
    
    if 'statistical_test' in results:
        print(f'\nStatistical Significance:')
        print(f'  p-value: {results["statistical_test"]["p_value"]:.6f}')
        print(f'  Cohen\'s d: {results["statistical_test"]["cohens_d"]:.4f}')
        print(f'  Effect size: {results["statistical_test"]["effect_size"]}')

## Performance by Salary Tier

Analyze model performance across different salary tiers.

In [ ]:
if 'error' not in results and 'tier_comparison' in results:
    tier_comparison = results['tier_comparison']
    
    print('Performance by Salary Tier:')
    print('='*80)
    print(tier_comparison[['salary_tier', 'count', 'model_mape', 'benchmark_mape', 'mape_improvement']].to_string(index=False))
    print('\nDetailed Breakdown:')
    for _, row in tier_comparison.iterrows():
        improvement = row['mape_improvement']
        status = 'BETTER' if improvement > 0 else 'WORSE'
        symbol = '+' if improvement > 0 else ''
        
        print(f'{str(row["salary_tier"]):20} {symbol}{improvement:6.1f}% {status:8} '
              f'(Model: {row["model_mape"]:.1f}%, Benchmark: {row["benchmark_mape"]:.1f}%)')
else:
    print('Tier comparison not available in results')

In [ ]:
pipeline = feature_config.build_pipeline(FeaturePipeline)

print(f'Feature pipeline configured with {len(pipeline.transformers)} transformers:')
for i, transformer in enumerate(pipeline.transformers, 1):
    print(f'  {i}. {transformer.__class__.__name__}')

print('\nBuilding features from training data...')
training_data_sorted = training_data.sort_values(['playerID', 'gameDate'])
training_features = pipeline.fit_transform(training_data_sorted)

print(f'Generated {len(training_features)} feature rows')
print(f'Feature columns: {len([col for col in training_features.columns if col.startswith(("rolling_", "ewma_"))])}')

print(f'\nFeature columns:')
feature_cols = [col for col in training_features.columns if col.startswith(("rolling_", "ewma_"))]
for col in sorted(feature_cols)[:20]:
    print(f'  {col}')
if len(feature_cols) > 20:
    print(f'  ... and {len(feature_cols) - 20} more')

In [ ]:
import pickle
import os

slate_dates = loader.load_slate_dates(TEST_START, TEST_END)

print(f'Found {len(slate_dates)} slates to backtest')
print(f'Date range: {slate_dates[0]} to {slate_dates[-1]}')

results_list = []
all_predictions = []

mape_metric = MAPEMetric()
rmse_metric = RMSEMetric()
mae_metric = MAEMetric()
corr_metric = CorrelationMetric()

for test_date in tqdm(slate_dates, desc='Backtesting slates'):
    print(f'\n{"="*60}')
    print(f'Processing Slate: {test_date}')
    print(f'{"="*60}')

    slate_data = loader.load_slate_data(test_date)
    salaries_df = slate_data.get('dfs_salaries', pd.DataFrame())

    if salaries_df.empty:
        logger.warning(f'No salary data for {test_date}, skipping')
        continue

    print(f'  Found {len(salaries_df)} players with salaries')

    # Create directory for this date's models if saving
    if SAVE_MODELS:
        date_models_dir = MODELS_DIR / 'per_player' / test_date
        date_models_dir.mkdir(parents=True, exist_ok=True)
        slate_models_dir = MODELS_DIR / 'per_slate'
        slate_models_dir.mkdir(parents=True, exist_ok=True)
        print(f'  Saving models to: {date_models_dir}')

    slate_predictions = []
    models_trained = 0
    models_saved = 0
    players_skipped_insufficient_data = 0
    
    # Collect all slate training data for slate-wide model
    slate_X_train_list = []
    slate_y_train_list = []
    slate_playerid_list = []

    for _, player_row in salaries_df.iterrows():
        player_id = player_row.get('playerID')
        player_name = player_row.get('longName', player_row.get('playerName', ''))

        player_training_data = training_features[training_features['playerID'] == player_id].copy()

        if len(player_training_data) < MIN_PLAYER_GAMES:
            players_skipped_insufficient_data += 1
            continue

        try:
            metadata_cols = ['playerID', 'longName', 'playerName', 'team', 'pos', 'gameDate', 'fpts']
            feature_cols = [col for col in player_training_data.columns if col not in metadata_cols and col.startswith(('rolling_', 'ewma_'))]

            X_train = player_training_data[feature_cols].fillna(0)
            y_train = player_training_data['fpts']

            if len(X_train) < 3 or y_train.isna().all():
                continue

            # Add to slate-wide training data
            slate_X_train_list.append(X_train)
            slate_y_train_list.append(y_train)
            slate_playerid_list.extend([player_id] * len(X_train))

            # Train per-player model
            model = XGBoostModel()
            model.train(X_train, y_train)
            models_trained += 1

            # Save the per-player model if configured
            if SAVE_MODELS:
                try:
                    # Clean player name for filename (remove special characters)
                    clean_name = ''.join(c if c.isalnum() or c in (' ', '-', '_') else '_' for c in player_name)
                    clean_name = clean_name.replace(' ', '_').lower()
                    
                    # Create filename with player ID and name
                    model_filename = f"{player_id}_{clean_name}.pkl"
                    model_path = date_models_dir / model_filename
                    
                    # Save model
                    model.save(str(model_path))
                    
                    # Also save feature columns for this model
                    feature_cols_path = date_models_dir / f"{player_id}_{clean_name}_features.pkl"
                    with open(feature_cols_path, 'wb') as f:
                        pickle.dump(feature_cols, f)
                    
                    models_saved += 1
                    
                except Exception as save_error:
                    logger.warning(f'Error saving model for {player_name}: {str(save_error)}')

            latest_features = X_train.iloc[[-1]]
            prediction = model.predict(latest_features)[0]

            slate_predictions.append({
                'date': test_date,
                'playerID': player_id,
                'playerName': player_name,
                'team': player_row.get('team', ''),
                'pos': player_row.get('pos', ''),
                'salary': player_row.get('salary', 0),
                'projected_fpts': prediction,
                'benchmark_pred': benchmark.player_averages.get(player_id, 0)
            })

        except Exception as e:
            logger.warning(f'Error training model for {player_name} on {test_date}: {str(e)}')
            continue

    # Train and save slate-wide model if we have data
    if SAVE_MODELS and slate_X_train_list:
        try:
            # Combine all player data for slate-wide model
            slate_X_combined = pd.concat(slate_X_train_list, ignore_index=True)
            slate_y_combined = pd.concat(slate_y_train_list, ignore_index=True)
            
            print(f'  Training slate-wide model with {len(slate_X_combined)} samples from {len(slate_X_train_list)} players')
            
            # Train slate-wide model
            slate_model = XGBoostModel()
            slate_model.train(slate_X_combined, slate_y_combined)
            
            # Save slate-wide model
            slate_model_path = slate_models_dir / f"{test_date}_slate_model.pkl"
            slate_model.save(str(slate_model_path))
            
            # Save feature columns for slate model
            slate_features_path = slate_models_dir / f"{test_date}_slate_features.pkl"
            with open(slate_features_path, 'wb') as f:
                pickle.dump(feature_cols, f)
            
            # Save player ID mapping for slate model
            slate_playerids_path = slate_models_dir / f"{test_date}_slate_playerids.pkl"
            with open(slate_playerids_path, 'wb') as f:
                pickle.dump(slate_playerid_list, f)
            
            print(f'  Saved slate-wide model to: {slate_model_path}')
            
        except Exception as slate_error:
            logger.warning(f'Error saving slate-wide model for {test_date}: {str(slate_error)}')

    print(f'  Models trained: {models_trained}')
    if SAVE_MODELS:
        print(f'  Per-player models saved: {models_saved}')
    print(f'  Players skipped (insufficient data): {players_skipped_insufficient_data}')

    if not slate_predictions:
        print(f'  WARNING: No predictions generated for this slate')
        continue

    predictions_df = pd.DataFrame(slate_predictions)
    print(f'  Generated {len(predictions_df)} predictions')

    # Save predictions to parquet for this slate
    if SAVE_MODELS:
        predictions_path = slate_models_dir / f"{test_date}.parquet"
        predictions_df.to_parquet(predictions_path)
        print(f'  Saved slate predictions to: {predictions_path}')

    filters = {'start_date': test_date, 'end_date': test_date}
    actuals_df = loader.storage.load('box_scores', filters)

    if actuals_df.empty:
        logger.warning(f'No actuals for {test_date}')
        continue

    actuals_df['actual_fpts'] = actuals_df.apply(calculate_dk_fantasy_points, axis=1)

    merged = predictions_df.merge(
        actuals_df[['playerID', 'actual_fpts']],
        on='playerID',
        how='inner'
    )

    if len(merged) == 0:
        print(f'  WARNING: No matching actual results found')
        continue

    print(f'  Matched {len(merged)} players with actual results')

    model_mape = mape_metric.calculate(merged['actual_fpts'], merged['projected_fpts'])
    model_rmse = rmse_metric.calculate(merged['actual_fpts'], merged['projected_fpts'])
    model_mae = mae_metric.calculate(merged['actual_fpts'], merged['projected_fpts'])
    model_corr = corr_metric.calculate(merged['actual_fpts'], merged['projected_fpts'])

    has_benchmark = (merged['benchmark_pred'] > 0)
    benchmark_mape = mape_metric.calculate(merged[has_benchmark]['actual_fpts'], merged[has_benchmark]['benchmark_pred']) if has_benchmark.any() else np.nan
    benchmark_rmse = rmse_metric.calculate(merged[has_benchmark]['actual_fpts'], merged[has_benchmark]['benchmark_pred']) if has_benchmark.any() else np.nan

    print(f'\n  Performance Metrics:')
    print(f'  {"="*40}')
    print(f'  Model Performance:')
    print(f'    MAPE: {model_mape:.2f}%')
    print(f'    RMSE: {model_rmse:.2f}')
    print(f'    MAE:  {model_mae:.2f}')
    print(f'    Correlation: {model_corr:.3f}')

    if not np.isnan(benchmark_mape):
        print(f'\n  Benchmark Performance:')
        print(f'    MAPE: {benchmark_mape:.2f}%')
        print(f'    RMSE: {benchmark_rmse:.2f}')
        print(f'\n  Improvement over Benchmark:')
        print(f'    MAPE: {benchmark_mape - model_mape:+.2f}% {"(Better)" if benchmark_mape > model_mape else "(Worse)"}')

    print(f'\n  Fantasy Points Summary:')
    print(f'    Mean Actual: {merged["actual_fpts"].mean():.2f}')
    print(f'    Mean Projected: {merged["projected_fpts"].mean():.2f}')
    print(f'    Mean Error: {(merged["projected_fpts"] - merged["actual_fpts"]).mean():+.2f}')

    # Analyze by salary tier for this date
    merged['salary'] = merged['salary'].astype(int)
    merged['salary_bin'] = pd.cut(merged['salary'], bins=SALARY_TIERS, labels=['Low', 'Mid', 'High', 'Elite'][:len(SALARY_TIERS)-1])

    print(f'\n  Performance by Salary Tier:')
    for tier in merged['salary_bin'].unique():
        if pd.isna(tier):
            continue
        tier_data = merged[merged['salary_bin'] == tier]
        if len(tier_data) > 0:
            tier_mape = mape_metric.calculate(tier_data['actual_fpts'], tier_data['projected_fpts'])
            print(f'    {tier}: MAPE={tier_mape:.1f}% (n={len(tier_data)})')

    # Save merged results with actuals for analysis
    if SAVE_MODELS:
        results_with_actuals_path = slate_models_dir / f"{test_date}_with_actuals.parquet"
        merged.to_parquet(results_with_actuals_path)

    results_list.append({
        'date': test_date,
        'num_players': len(merged),
        'models_trained': models_trained,
        'models_saved': models_saved if SAVE_MODELS else 0,
        'model_mape': model_mape,
        'model_rmse': model_rmse,
        'model_mae': model_mae,
        'model_corr': model_corr,
        'benchmark_mape': benchmark_mape,
        'benchmark_rmse': benchmark_rmse,
        'mean_actual': merged['actual_fpts'].mean(),
        'mean_projected': merged['projected_fpts'].mean(),
        'mean_benchmark': merged['benchmark_pred'].mean()
    })

    all_predictions.append(merged)

results_df = pd.DataFrame(results_list)
all_predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()

print(f'\n{"="*60}')
print(f'Completed backtest for {len(results_df)} slates')
print(f'Total predictions: {len(all_predictions_df)}')
if SAVE_MODELS:
    print(f'Total per-player models saved: {results_df["models_saved"].sum():.0f}')
    print(f'Models directory: {MODELS_DIR}')
print(f'{"="*60}')

In [ ]:
comparison_df = all_predictions_df[(all_predictions_df['projected_fpts'] > 0) & (all_predictions_df['benchmark_pred'] > 0)].copy()

comparison_results = benchmark.compare_with_model(
    actual=comparison_df['actual_fpts'],
    model_pred=comparison_df['projected_fpts'],
    benchmark_pred=comparison_df['benchmark_pred']
)

print(comparison_results['summary'])

In [ ]:
# Use plotly dark theme and vibrant custom colors for all traces and lines.

import plotly.io as pio
pio.templates.default = "plotly_dark"

vibrant_colors = {
    "model": "#9AFF6E",        # Vibrant light green
    "benchmark": "#3ABEFF",    # Vibrant blue/cyan
    "model_mean": "#FFFF35",   # Lime Yellow
    "benchmark_mean": "#FD5A66", # Vibrant Red
    "rmse": "#00FFC2",         # Aqua
    "rmse_mean": "#EA00FF",    # Vibrant Magenta
    "corr": "#FBBF24",         # Vibrant Gold
    "corr_mean": "#FF7A00",    # Orange
    "players_bar": "#FF53A1",  # Hot Pink
    "players_mean": "#25FFF1", # Electric Cyan
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '<b>MAPE Over Time</b>', 
        '<b>RMSE Over Time</b>', 
        '<b>Correlation Over Time</b>', 
        '<b>Players Evaluated Per Slate</b>'),
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

# MAPE Over Time
fig.add_trace(
    go.Scatter(
        x=results_df['date'], 
        y=results_df['model_mape'], 
        mode='lines+markers', 
        name='Model', 
        line=dict(width=2, color=vibrant_colors["model"]), 
        marker=dict(size=10, color=vibrant_colors["model"], symbol='circle', line=dict(width=1, color='black'))
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=results_df['date'], 
        y=results_df['benchmark_mape'], 
        mode='lines+markers', 
        name='Benchmark', 
        line=dict(width=2, color=vibrant_colors["benchmark"]),
        marker=dict(size=10, color=vibrant_colors["benchmark"], symbol='square', line=dict(width=1, color='black')),
        opacity=0.85
    ),
    row=1, col=1
)
fig.add_hline(
    y=results_df['model_mape'].mean(),
    line_dash="dash",
    line_color=vibrant_colors["model_mean"],
    opacity=0.7,
    row=1, col=1,
    annotation_text="Model Mean",
    annotation_position="right"
)
fig.add_hline(
    y=results_df['benchmark_mape'].mean(),
    line_dash="dash",
    line_color=vibrant_colors["benchmark_mean"],
    opacity=0.7,
    row=1, col=1,
    annotation_text="Benchmark Mean",
    annotation_position="right"
)

# RMSE Over Time  
fig.add_trace(
    go.Scatter(
        x=results_df['date'], 
        y=results_df['model_rmse'], 
        mode='lines+markers', 
        name='RMSE',
        line=dict(color=vibrant_colors["rmse"], width=2), 
        marker=dict(size=10, color=vibrant_colors["rmse"], symbol='diamond', line=dict(width=1, color='black'))
    ),
    row=1, col=2
)
fig.add_hline(
    y=results_df['model_rmse'].mean(), 
    line_dash="dash", 
    line_color=vibrant_colors["rmse_mean"], 
    row=1, col=2,
    annotation_text=f"Mean: {results_df['model_rmse'].mean():.2f}",
    annotation_position="right"
)

# Correlation Over Time
fig.add_trace(
    go.Scatter(
        x=results_df['date'], 
        y=results_df['model_corr'], 
        mode='lines+markers', 
        name='Correlation',
        line=dict(color=vibrant_colors["corr"], width=2), 
        marker=dict(size=10, color=vibrant_colors["corr"], symbol='cross', line=dict(width=1, color='black'))
    ),
    row=2, col=1
)
fig.add_hline(
    y=results_df['model_corr'].mean(), 
    line_dash="dash", 
    line_color=vibrant_colors["corr_mean"], 
    row=2, col=1,
    annotation_text=f"Mean: {results_df['model_corr'].mean():.3f}",
    annotation_position="right"
)

# Players Evaluated Per Slate
fig.add_trace(
    go.Bar(
        x=results_df['date'], 
        y=results_df['num_players'], 
        name='Players', 
        marker=dict(color=vibrant_colors["players_bar"], opacity=0.85, line=dict(color="white", width=0.5))
    ),
    row=2, col=2
)
fig.add_hline(
    y=results_df['num_players'].mean(), 
    line_dash="dash", 
    line_color=vibrant_colors["players_mean"], 
    row=2, col=2,
    annotation_text=f"Mean: {results_df['num_players'].mean():.1f}",
    annotation_position="right"
)

# Update layout for dark theme and axis/legend colors
axis_style = dict(color="white", showline=True, linewidth=1.5, linecolor='#666', zerolinecolor="#444")
fig.update_xaxes(title_text="<b>Date</b>", row=1, col=1, tickangle=45, **axis_style)
fig.update_xaxes(title_text="<b>Date</b>", row=1, col=2, tickangle=45, **axis_style)
fig.update_xaxes(title_text="<b>Date</b>", row=2, col=1, tickangle=45, **axis_style)
fig.update_xaxes(title_text="<b>Date</b>", row=2, col=2, tickangle=45, **axis_style)

fig.update_yaxes(title_text="<b>MAPE (%)</b>", row=1, col=1, **axis_style)
fig.update_yaxes(title_text="<b>RMSE</b>", row=1, col=2, **axis_style)
fig.update_yaxes(title_text="<b>Correlation</b>", row=2, col=1, **axis_style)
fig.update_yaxes(title_text="<b>Number of Players</b>", row=2, col=2, **axis_style)

fig.update_layout(
    height=800,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="center",
        x=0.5,
        font=dict(color="white", size=14)
    ),
    plot_bgcolor="#22272e",
    paper_bgcolor="#22272e",
    font=dict(family="Segoe UI, Roboto, Arial", size=14, color="white"),
    title_text="<b>Backtest Performance Metrics</b>",
    title_x=0.5,
    margin=dict(l=40, r=40, t=40, b=40)  # Reduced top margin to prevent title overlap
)

# Move the title lower by adding extra top padding to the subplot area, 
# so title does not overlap with subplot titles or legend.
fig.update_layout(
    margin=dict(t=110)  # Increase only top margin for extra spacing for title
)

fig.show()

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('MAPE by Salary Tier', 'Model Improvement Over Benchmark<br>(Positive = Model Better)'),
    horizontal_spacing=0.15
)

# MAPE by Salary Tier - Grouped Bar Chart
x_labels = tier_comparison['salary_tier'].astype(str)
x_pos = np.arange(len(x_labels))

fig.add_trace(
    go.Bar(x=x_labels, y=tier_comparison['model_mape'],
           name='Model', marker=dict(opacity=0.8)),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=x_labels, y=tier_comparison['benchmark_mape'],
           name='Benchmark', marker=dict(opacity=0.8)),
    row=1, col=1
)

# Improvement Bar Chart
colors = ['green' if x > 0 else 'red' for x in tier_comparison['mape_improvement']]
fig.add_trace(
    go.Bar(x=x_labels, y=tier_comparison['mape_improvement'],
           marker=dict(color=colors, opacity=0.7),
           showlegend=False),
    row=1, col=2
)

# Add horizontal line at 0 for improvement chart
fig.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=2)

# Update layout
fig.update_xaxes(title_text="Salary Tier", row=1, col=1)
fig.update_xaxes(title_text="Salary Tier", row=1, col=2)
fig.update_yaxes(title_text="MAPE (%)", row=1, col=1)
fig.update_yaxes(title_text="MAPE Improvement (%)", row=1, col=2)

fig.update_layout(
    height=500,
    barmode='group',
    title_text="Performance Analysis by Salary Tier",
    title_x=0.5
)

fig.show()

In [ ]:
if 'error' not in results and 'benchmark_comparison' in results:
    comparison_results = results['benchmark_comparison']
    print(comparison_results['summary'])
else:
    print('Benchmark comparison not available')

In [ ]:
if 'error' not in results and 'statistical_test' in results:
    stat_test = results['statistical_test']
    
    print('Statistical Significance Test (Paired t-test):')
    print(f'  t-statistic: {stat_test["t_statistic"]:.4f}')
    print(f'  p-value: {stat_test["p_value"]:.6f}')
    print()
    
    if stat_test['p_value'] < 0.05:
        if stat_test['t_statistic'] < 0:
            print('  Result: Model is SIGNIFICANTLY BETTER than benchmark (p < 0.05)')
        else:
            print('  Result: Model is SIGNIFICANTLY WORSE than benchmark (p < 0.05)')
    else:
        print('  Result: No significant difference between model and benchmark (p >= 0.05)')
    
    print(f'\n  Cohen\'s d: {stat_test["cohens_d"]:.4f}')
    print(f'  Effect size: {stat_test["effect_size"]}')
else:
    print('Statistical test not available')

In [ ]:
if 'error' not in results:
    # Use plotly dark theme and vibrant custom colors for all traces and lines.
    
    import plotly.io as pio
    pio.templates.default = "plotly_dark"
    
    vibrant_colors = {
        "model": "#9AFF6E",        # Vibrant light green
        "benchmark": "#3ABEFF",    # Vibrant blue/cyan
        "model_mean": "#FFFF35",   # Lime Yellow
        "benchmark_mean": "#FD5A66", # Vibrant Red
        "rmse": "#00FFC2",         # Aqua
        "rmse_mean": "#EA00FF",    # Vibrant Magenta
        "corr": "#FBBF24",         # Vibrant Gold
        "corr_mean": "#FF7A00",    # Orange
        "players_bar": "#FF53A1",  # Hot Pink
        "players_mean": "#25FFF1", # Electric Cyan
    }
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            '<b>MAPE Over Time</b>', 
            '<b>RMSE Over Time</b>', 
            '<b>Correlation Over Time</b>', 
            '<b>Players Evaluated Per Slate</b>'),
        vertical_spacing=0.15,
        horizontal_spacing=0.1
    )
    
    # MAPE Over Time
    fig.add_trace(
        go.Scatter(
            x=results_df['date'], 
            y=results_df['model_mape'], 
            mode='lines+markers', 
            name='Model', 
            line=dict(width=2, color=vibrant_colors["model"]), 
            marker=dict(size=10, color=vibrant_colors["model"], symbol='circle', line=dict(width=1, color='black'))
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=results_df['date'], 
            y=results_df['benchmark_mape'], 
            mode='lines+markers', 
            name='Benchmark', 
            line=dict(width=2, color=vibrant_colors["benchmark"]),
            marker=dict(size=10, color=vibrant_colors["benchmark"], symbol='square', line=dict(width=1, color='black')),
            opacity=0.85
        ),
        row=1, col=1
    )
    fig.add_hline(
        y=results_df['model_mape'].mean(),
        line_dash="dash",
        line_color=vibrant_colors["model_mean"],
        opacity=0.7,
        row=1, col=1,
        annotation_text="Model Mean",
        annotation_position="right"
    )
    fig.add_hline(
        y=results_df['benchmark_mape'].mean(),
        line_dash="dash",
        line_color=vibrant_colors["benchmark_mean"],
        opacity=0.7,
        row=1, col=1,
        annotation_text="Benchmark Mean",
        annotation_position="right"
    )
    
    # RMSE Over Time  
    fig.add_trace(
        go.Scatter(
            x=results_df['date'], 
            y=results_df['model_rmse'], 
            mode='lines+markers', 
            name='RMSE',
            line=dict(color=vibrant_colors["rmse"], width=2), 
            marker=dict(size=10, color=vibrant_colors["rmse"], symbol='diamond', line=dict(width=1, color='black'))
        ),
        row=1, col=2
    )
    fig.add_hline(
        y=results_df['model_rmse'].mean(), 
        line_dash="dash", 
        line_color=vibrant_colors["rmse_mean"], 
        row=1, col=2,
        annotation_text=f"Mean: {results_df['model_rmse'].mean():.2f}",
        annotation_position="right"
    )
    
    # Correlation Over Time
    fig.add_trace(
        go.Scatter(
            x=results_df['date'], 
            y=results_df['model_corr'], 
            mode='lines+markers', 
            name='Correlation',
            line=dict(color=vibrant_colors["corr"], width=2), 
            marker=dict(size=10, color=vibrant_colors["corr"], symbol='cross', line=dict(width=1, color='black'))
        ),
        row=2, col=1
    )
    fig.add_hline(
        y=results_df['model_corr'].mean(), 
        line_dash="dash", 
        line_color=vibrant_colors["corr_mean"], 
        row=2, col=1,
        annotation_text=f"Mean: {results_df['model_corr'].mean():.3f}",
        annotation_position="right"
    )
    
    # Players Evaluated Per Slate
    fig.add_trace(
        go.Bar(
            x=results_df['date'], 
            y=results_df['num_players'], 
            name='Players', 
            marker=dict(color=vibrant_colors["players_bar"], opacity=0.85, line=dict(color="white", width=0.5))
        ),
        row=2, col=2
    )
    fig.add_hline(
        y=results_df['num_players'].mean(), 
        line_dash="dash", 
        line_color=vibrant_colors["players_mean"], 
        row=2, col=2,
        annotation_text=f"Mean: {results_df['num_players'].mean():.1f}",
        annotation_position="right"
    )
    
    # Update layout for dark theme and axis/legend colors
    axis_style = dict(color="white", showline=True, linewidth=1.5, linecolor='#666', zerolinecolor="#444")
    fig.update_xaxes(title_text="<b>Date</b>", row=1, col=1, tickangle=45, **axis_style)
    fig.update_xaxes(title_text="<b>Date</b>", row=1, col=2, tickangle=45, **axis_style)
    fig.update_xaxes(title_text="<b>Date</b>", row=2, col=1, tickangle=45, **axis_style)
    fig.update_xaxes(title_text="<b>Date</b>", row=2, col=2, tickangle=45, **axis_style)
    
    fig.update_yaxes(title_text="<b>MAPE (%)</b>", row=1, col=1, **axis_style)
    fig.update_yaxes(title_text="<b>RMSE</b>", row=1, col=2, **axis_style)
    fig.update_yaxes(title_text="<b>Correlation</b>", row=2, col=1, **axis_style)
    fig.update_yaxes(title_text="<b>Number of Players</b>", row=2, col=2, **axis_style)
    
    fig.update_layout(
        height=800,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.05,
            xanchor="center",
            x=0.5,
            font=dict(color="white", size=14)
        ),
        plot_bgcolor="#22272e",
        paper_bgcolor="#22272e",
        font=dict(family="Segoe UI, Roboto, Arial", size=14, color="white"),
        title_text="<b>Backtest Performance Metrics</b>",
        title_x=0.5,
        margin=dict(l=40, r=40, t=110, b=40)
    )
    
    fig.show()

In [ ]:
if 'error' not in results and not all_predictions_df.empty:
    # Filter for valid comparisons
    comparison_df = all_predictions_df[
        (all_predictions_df['projected_fpts'] > 0) & 
        (all_predictions_df['benchmark_pred'] > 0)
    ].copy()
    
    # Dark theme and vibrant colors
    vibrant_colors = {
        "scatter": "#00D7FF",        # Cyan
        "diagonal": "#FF0080",       # Pink/magenta
        "histogram": "#18FF6D",      # Vibrant green
        "histogram_border": "#22272e",
        "vline_zero": "#FF0080",     # Pink/magenta
        "vline_mean": "#FFD700",     # Gold/yellow
    }
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            '<span style="color:white"><b>Model vs Benchmark Error Comparison</b></span>', 
            '<span style="color:white"><b>Error Difference Distribution</b><br>(Positive = Model Better)</span>'
        ),
        horizontal_spacing=0.15
    )
    
    # Calculate errors
    comparison_df['model_error'] = np.abs(comparison_df['projected_fpts'] - comparison_df['actual_fpts'])
    comparison_df['benchmark_error'] = np.abs(comparison_df['benchmark_pred'] - comparison_df['actual_fpts'])
    
    # Scatter plot: Model vs Benchmark Error
    fig.add_trace(
        go.Scatter(
            x=comparison_df['benchmark_error'], y=comparison_df['model_error'],
            mode='markers',
            marker=dict(size=7, opacity=0.7, color=vibrant_colors["scatter"], line=dict(width=0)),
            name='Errors',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Add diagonal line for equal error
    max_error = max(comparison_df['benchmark_error'].max(), comparison_df['model_error'].max()) * 1.03
    fig.add_trace(
        go.Scatter(
            x=[0, max_error], y=[0, max_error],
            mode='lines',
            line=dict(color=vibrant_colors["diagonal"], dash='dash', width=2),
            name='Equal error',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Histogram of error differences
    error_diff = comparison_df['benchmark_error'] - comparison_df['model_error']
    fig.add_trace(
        go.Histogram(
            x=error_diff, nbinsx=30,
            marker=dict(color=vibrant_colors["histogram"], opacity=0.85, 
                        line=dict(color=vibrant_colors["histogram_border"], width=1.2)),
            name='Error Difference',
            showlegend=True
        ),
        row=1, col=2
    )
    
    # Add vertical lines for reference
    fig.add_vline(
        x=0,
        line_dash="dash",
        line_color=vibrant_colors["vline_zero"],
        row=1, col=2,
        annotation_text="<b style='color:#FF0080'>No difference</b>",
        annotation_position="top"
    )
    fig.add_vline(
        x=error_diff.mean(),
        line_dash="dash",
        line_color=vibrant_colors["vline_mean"],
        row=1, col=2,
        annotation_text=f"<b style='color:#FFD700'>Mean: {error_diff.mean():.2f}</b>",
        annotation_position="top right"
    )
    
    # Update layout for dark theme
    axis_style = dict(color="white", showline=True, linewidth=1.7, linecolor='#666', zerolinecolor="#444")
    fig.update_xaxes(title_text="<b style='color:#75eaff'>Benchmark Error</b>", row=1, col=1, tickfont_color="white", **axis_style)
    fig.update_yaxes(title_text="<b style='color:#FF0080'>Model Error</b>", row=1, col=1, tickfont_color="white", **axis_style)
    fig.update_xaxes(title_text="<b style='color:#18FF6D'>Error Difference (Benchmark - Model)</b>", row=1, col=2, tickfont_color="white", **axis_style)
    fig.update_yaxes(title_text="<b style='color:#FFD700'>Frequency</b>", row=1, col=2, tickfont_color="white", **axis_style)
    
    fig.update_layout(
        height=800,
        width=1400,
        showlegend=True,
        legend=dict(
            orientation="h",
            y=1.05,
            yanchor="bottom",
            xanchor="center",
            x=0.5,
            font=dict(color="white", size=13)
        ),
        plot_bgcolor="#22272e",
        paper_bgcolor="#22272e",
        font=dict(family="Segoe UI, Roboto, Arial", size=15, color="white"),
        title_text="<b>Model vs Benchmark Error Analysis</b>",
        title_x=0.5,
        margin=dict(l=60, r=60, t=80, b=55)
    )
    
    fig.show()